# Real-Speech Serving: Five-Stage Optimization Comparison

This notebook compares the complete progression on the same 100 LibriSpeech utterances under **burst** and **Poisson-high (4 requests/s)** traffic:

1. `sequential`
2. `static_batching`
3. `hybrid_engine` — paged continuous batching with per-step cross-KV gathers
4. `hybrid_one_time_cross_kv` — cross-KV gathered once at handoff and reused
5. `hybrid_graph_no_host_sync` — one-time cross-KV plus removed per-step host tensor creation/CPU token sync, CUDA graph replay, GPU argmax, persistent buffers, and K=4 readback

Every implementation must first pass the 100-utterance parity gate. A failing implementation is not benchmarked.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts" / "run_five_system_comparison.py").exists():
    raise RuntimeError("Launch Jupyter from the repository root")

PYTHON = sys.executable
OUTPUT_JSON = REPO_ROOT / "data" / "real_speech_five_system_comparison.json"

# full runs the complete comparison. cached only renders an existing result.
RUN_MODE = os.environ.get("REAL_SPEECH_RUN_MODE", "full")
REPEATS = 3
assert RUN_MODE in {"full", "cached"}
print("Run mode:", RUN_MODE)
print("Measured repeats:", REPEATS)
print("Python:", PYTHON)


Run mode: full
Measured repeats: 3
Python: c:\Users\Aditya Kudale\AppData\Local\Programs\Python\Python310\python.exe


## 1. Environment

The benchmark uses the current notebook kernel's Python environment. Confirm that CUDA and the pinned package versions are active before starting the long run.


In [2]:
import torch
import transformers
import triton

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Triton:", triton.__version__)
subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu,clocks.gr,power.draw", "--format=csv,noheader"], check=False)


CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Torch: 2.6.0+cu124
Transformers: 4.57.0
Triton: 3.8.0


CompletedProcess(args=['nvidia-smi', '--query-gpu=temperature.gpu,clocks.gr,power.draw', '--format=csv,noheader'], returncode=0)

## 2. Confirm the five implementations

These are separate runtime configurations, not renamed copies. In particular, the legacy hybrid engine gathers encoder cross-KV from paged blocks on every decode step, while the one-time version caches it at handoff. The final graph version additionally removes per-step host tensor construction and per-token CPU synchronization.


In [3]:
from scripts.run_five_system_comparison import DESCRIPTIONS, SYSTEMS

for index, (name, options) in enumerate(SYSTEMS, 1):
    print(f"{index}. {name}")
    print("   options:", options)
    print("   ", DESCRIPTIONS[name])


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

1. sequential
   options: {'kind': 'sequential'}
    One request at a time; no batching or paged scheduler.
2. static_batching
   options: {'kind': 'static'}
    Fixed batches of four; waits for the slowest request in each batch.
3. hybrid_engine
   options: {'kind': 'engine', 'cache_cross_kv': False, 'graph_decode': False}
    Continuous batching with paged dual KV; cross-attention KV is gathered from blocks every decode step.
4. hybrid_one_time_cross_kv
   options: {'kind': 'engine', 'cache_cross_kv': True, 'graph_decode': False}
    Hybrid engine with cross-attention KV gathered once at encode-to-decode handoff and reused.
5. hybrid_graph_no_host_sync
   options: {'kind': 'engine', 'cache_cross_kv': True, 'graph_decode': True}
    One-time cross-KV plus CUDA-graph decode, persistent inputs, GPU argmax, event-owned pinned input ring, and K=4 host readback.


## 3. Execute parity gate and measured comparison

The runner performs:

- 100-utterance burst parity preflight for all five systems;
- a discarded warm-up for every system;
- three measured repeats of burst and Poisson-high traffic;
- cyclic system-order rotation to reduce systematic thermal/order bias;
- median latency, throughput, capacity, VRAM, WER, and exact-match aggregation.

Expect this cell to take a substantial amount of time.


In [4]:
if RUN_MODE == "full":
    subprocess.run(
        [PYTHON, "-u", "scripts/run_five_system_comparison.py", "--repeats", str(REPEATS)],
        cwd=REPO_ROOT,
        check=True,
    )
elif not OUTPUT_JSON.exists():
    raise FileNotFoundError("No cached result exists. Set RUN_MODE='full'.")

results = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
print("Result status:", results["status"])
assert results["status"] == "VALID", results.get("invalid_systems")


Result status: VALID


## 4. Parity gate

Exact match must be at least 99%, and WER must remain within 0.2 percentage points of the Hugging Face reference.


In [5]:
import pandas as pd

parity_rows = []
for name, gate in results["parity_gate"].items():
    parity_rows.append({"system": name, **gate})
parity_table = pd.DataFrame(parity_rows)
display(parity_table)


,system,passed,exact_match_pct,wer_pct,reference_wer_pct,wer_delta_points
0,sequential,True,100.0,5.281536,5.281536,0.0
1,static_batching,True,100.0,5.281536,5.281536,0.0
2,hybrid_engine,True,99.0,5.281536,5.281536,0.0
3,hybrid_one_time_cross_kv,True,99.0,5.281536,5.281536,0.0
4,hybrid_graph_no_host_sync,True,99.0,5.281536,5.281536,0.0


## 5. Real-speech comparison tables

Each row is the median of the measured repeats. Burst service capacity is `100 / wall_time`; Poisson throughput is arrival-limited, so latency and queueing are the important measures there.


In [6]:
comparison_rows = []
for profile_name, profile in results["profiles"].items():
    for system_name, metrics in profile["systems"].items():
        comparison_rows.append({
            "profile": profile_name,
            "system": system_name,
            "capacity_req_s": metrics["service_capacity_req_per_sec_median"],
            "audio_throughput_x": metrics["throughput_audio_per_sec_median"],
            "wall_time_s": metrics["wall_time_s_median"],
            "TTFT_p50_s": metrics["ttft_p50_s_median"],
            "TTFT_p95_s": metrics["ttft_p95_s_median"],
            "latency_p50_s": metrics["latency_p50_s_median"],
            "latency_p95_s": metrics["latency_p95_s_median"],
            "peak_VRAM_MB": metrics["peak_vram_allocated_mb_median"],
            "WER_pct": metrics["wer_pct"],
            "exact_match_pct": metrics["exact_match_pct"],
        })

comparison = pd.DataFrame(comparison_rows)
for profile_name in results["profiles"]:
    print("\n", profile_name.upper())
    display(comparison[comparison["profile"] == profile_name].reset_index(drop=True))



 BURST


,profile,system,capacity_req_s,audio_throughput_x,wall_time_s,TTFT_p50_s,TTFT_p95_s,latency_p50_s,latency_p95_s,peak_VRAM_MB,WER_pct,exact_match_pct
0,burst,sequential,1.958220,17.265230,51.066797,24.073830,48.817647,24.633137,48.930622,221.960449,5.281536,100.0
1,burst,static_batching,4.801226,42.331452,20.828012,10.201284,19.564045,10.904013,19.859462,336.736816,5.281536,100.0
2,burst,hybrid_engine,2.192348,19.329497,45.613188,22.326122,42.966568,24.008873,43.897029,1785.934570,5.281536,99.0
3,burst,hybrid_one_time_cross_kv,2.664170,23.489456,37.535140,17.711318,34.989680,18.896293,35.939674,1785.934570,5.281536,99.0
4,burst,hybrid_graph_no_host_sync,5.719813,50.430446,17.483090,8.728239,16.599963,9.282341,17.087034,1926.721680,5.281536,99.0



 POISSON_HIGH_4_0


,profile,system,capacity_req_s,audio_throughput_x,wall_time_s,TTFT_p50_s,TTFT_p95_s,latency_p50_s,latency_p95_s,peak_VRAM_MB,WER_pct,exact_match_pct
0,poisson_high_4_0,sequential,2.164614,19.084971,46.197608,11.244214,21.952670,11.704218,22.269184,246.335449,5.281536,100.0
1,poisson_high_4_0,static_batching,4.159123,36.670154,24.043531,0.679271,1.670611,1.388809,2.278541,369.518066,5.281536,100.0
2,poisson_high_4_0,hybrid_engine,2.130766,18.786541,46.931471,12.622808,22.906622,14.015616,23.962569,1818.196289,5.281536,99.0
3,poisson_high_4_0,hybrid_one_time_cross_kv,2.462991,21.715700,40.601040,8.171329,16.226466,9.039271,17.286421,1812.112793,5.281536,99.0
4,poisson_high_4_0,hybrid_graph_no_host_sync,4.321819,38.104617,23.138404,0.154555,0.415515,0.421387,0.996532,1951.088867,5.281536,99.0


## 6. Incremental effect of each optimization

This table makes the requested progression explicit. Positive capacity change and negative latency change are improvements.


In [7]:
system_order = [name for name, _ in SYSTEMS]
delta_rows = []
for profile_name, profile in results["profiles"].items():
    previous_name = None
    for system_name in system_order:
        current = profile["systems"][system_name]
        if previous_name is not None:
            previous = profile["systems"][previous_name]
            delta_rows.append({
                "profile": profile_name,
                "transition": f"{previous_name} -> {system_name}",
                "capacity_change_pct": (
                    current["service_capacity_req_per_sec_median"] / previous["service_capacity_req_per_sec_median"] - 1
                ) * 100,
                "latency_p50_change_pct": (
                    current["latency_p50_s_median"] / previous["latency_p50_s_median"] - 1
                ) * 100,
                "latency_p95_change_pct": (
                    current["latency_p95_s_median"] / previous["latency_p95_s_median"] - 1
                ) * 100,
                "VRAM_change_MB": current["peak_vram_allocated_mb_median"] - previous["peak_vram_allocated_mb_median"],
            })
        previous_name = system_name

display(pd.DataFrame(delta_rows))


,profile,transition,capacity_change_pct,latency_p50_change_pct,latency_p95_change_pct,VRAM_change_MB
0,burst,sequential -> static_batching,145.183250,-55.734372,-59.413019,114.776367
1,burst,static_batching -> hybrid_engine,-54.337741,120.183841,121.038356,1449.197754
2,burst,hybrid_engine -> hybrid_one_time_cross_kv,21.521296,-21.294546,-18.127319,0.000000
3,burst,hybrid_one_time_cross_kv -> hybrid_graph_no_ho...,114.693973,-50.877448,-52.456348,140.787109
4,poisson_high_4_0,sequential -> static_batching,92.141527,-88.134113,-89.768187,123.182617
5,poisson_high_4_0,static_batching -> hybrid_engine,-48.768853,909.182158,951.662749,1448.678223
6,poisson_high_4_0,hybrid_engine -> hybrid_one_time_cross_kv,15.591796,-35.505718,-27.860736,-6.083496
7,poisson_high_4_0,hybrid_one_time_cross_kv -> hybrid_graph_no_ho...,75.470359,-95.338259,-94.235177,138.976074


## Interpretation

The comparison that directly answers Nathan's question is:

- `hybrid_one_time_cross_kv` versus `hybrid_graph_no_host_sync`

They share the one-time cross-KV optimization; the principal difference is removal of per-step host tensor construction and per-token CPU synchronization through persistent CUDA-graph buffers and K=4 readback. Use burst capacity for saturation behavior and Poisson-high latency for queueing impact.
